In [63]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

lat, lon = 22.5626, 88.3630

# ── Date range: past 23 hours ──────────────────────────────
now          = datetime.now()
start        = now - timedelta(hours=23)
start_date   = start.strftime("%Y-%m-%d")
end_date     = now.strftime("%Y-%m-%d")

# ── Fetch air quality ──────────────────────────────────────
aq_url = (
    f"https://air-quality-api.open-meteo.com/v1/air-quality"
    f"?latitude={lat}&longitude={lon}"
    f"&hourly=pm2_5,pm10,ozone,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide"
    f"&start_date={start_date}&end_date={end_date}"
    f"&timezone=Asia%2FKolkata"
)

# ── Fetch weather ──────────────────────────────────────────
weather_url = (
    f"https://api.open-meteo.com/v1/forecast"
    f"?latitude={lat}&longitude={lon}"
    f"&hourly=temperature_2m,relative_humidity_2m,rain,wind_speed_10m"
    f"&wind_speed_unit=ms"
    f"&start_date={start_date}&end_date={end_date}"
    f"&timezone=Asia%2FKolkata"
)

df_aq      = pd.DataFrame(requests.get(aq_url).json()['hourly'])
df_weather = pd.DataFrame(requests.get(weather_url).json()['hourly'])

# ── Merge ──────────────────────────────────────────────────
df = pd.merge(df_aq, df_weather, on='time')

# ── Rename columns ─────────────────────────────────────────
df.rename(columns={
    'pm2_5'              : 'pm25',
    'pm10'               : 'pm10',
    'ozone'              : 'o3',
    'carbon_monoxide'    : 'co',
    'nitrogen_dioxide'   : 'no2',
    'sulphur_dioxide'    : 'so2',
    'temperature_2m'     : 'temp',
    'relative_humidity_2m': 'rh',
    'rain'               : 'rain',
    'wind_speed_10m'     : 'wind'
}, inplace=True)

# ── Filter to last 23 hours up to current hour ─────────────
current_hour_str = now.strftime("%Y-%m-%dT%H:00")
df['time']       = pd.to_datetime(df['time'])
cutoff           = pd.to_datetime(current_hour_str)
df        = df[df['time'] <= cutoff].tail(23).reset_index(drop=True)
df

,time,pm25,pm10,o3,co,no2,so2,temp,rh,rain,wind
0,2026-06-17 21:00:00,43.2,51.6,60.0,428.0,14.8,9.8,30.4,81,0.0,3.05
1,2026-06-17 22:00:00,45.3,55.2,58.0,394.0,14.2,8.6,30.1,85,0.0,2.97
2,2026-06-17 23:00:00,46.3,57.3,58.0,362.0,13.5,7.7,30.6,86,0.0,2.64
3,2026-06-18 00:00:00,46.4,58.2,59.0,331.0,12.4,7.1,30.3,86,0.0,2.56
4,2026-06-18 01:00:00,45.6,57.7,61.0,303.0,11.2,6.6,30.2,85,0.0,2.86
5,2026-06-18 02:00:00,45.2,57.2,63.0,292.0,10.4,6.4,30.0,87,0.0,2.08
6,2026-06-18 03:00:00,45.2,57.3,62.0,310.0,10.8,6.5,29.8,89,0.0,2.46
7,2026-06-18 04:00:00,46.1,58.4,60.0,345.0,11.8,6.8,29.6,90,0.0,2.22
8,2026-06-18 05:00:00,48.1,61.5,72.0,383.0,10.1,7.5,30.1,90,0.0,2.15
9,2026-06-18 06:00:00,48.4,62.6,88.0,428.0,8.5,8.7,31.0,87,0.0,2.11


In [64]:
df['co'] = df['co'] / 1000

In [65]:
import joblib
scaler = joblib.load('scaler.pkl')

In [66]:
model = keras.models.load_model(
    'model_3.keras',
    custom_objects={'ILSTMCell': ILSTMCell, 'ILSTMLayer': ILSTMLayer}
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/layer.py:427: UserWarning: `build()` was called on layer 'ilstm', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


In [67]:
df["time"] = pd.to_datetime(df["time"])
df["year"]  = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"]   = df["time"].dt.day
df["hour"]  = df["time"].dt.hour

In [68]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

In [69]:
def calculate_heat_index(T, RH):
    HI = (-8.78
          + 1.61 * T
          + 2.34 * RH
          - 0.146 * T * RH
          - 0.012 * T**2
          - 0.016 * RH**2
          + 0.00022 * T**2 * RH
          + 0.00086 * T * RH**2
          - 0.000002 * T**2 * RH**2)
    return HI

df["heat_index"] = calculate_heat_index(df["temp"], df["rh"])

In [70]:
import numpy as np
df["pm25"] = np.log1p(df["pm25"])
df["pm10"] = np.log1p(df["pm10"])

In [71]:
festival_dates = [
    # Diwali / Kali Puja
    "2026-11-08",
    # Dussehra
    "2026-10-20"
]

festival_dates = pd.to_datetime(festival_dates).date

df["festival_flag"] = df["time"].dt.date.isin(festival_dates).astype(int)

In [72]:
# Morning rush: 7am-10am, Evening rush: 5pm-8pm
rush_hours = list(range(7, 11)) + list(range(17, 21))

df["rush_hour_flag"] = df["hour"].isin(rush_hours).astype(int)

In [73]:
def get_season(month):
    if month in [12, 1, 2]:
        return 0   # Winter
    elif month in [3, 4, 5]:
        return 1   # Summer
    elif month in [6, 7, 8, 9]:
        return 2   # Monsoon
    else:
        return 3   # Post-monsoon (Oct, Nov)

df["season_flag"] = df["month"].apply(get_season)

In [74]:
df["crop_burning_flag"] = (
    (df["month"].isin([10, 11])) & 
    ~((df["month"] == 10) & (df["day"] < 15))
).astype(int)

In [77]:
df.drop(columns=['time','year','month','day','hour'],inplace=True)

In [89]:
cols_to_scale = ['pm25', 'pm10', 'no2', 'co', 'so2', 'o3', 
                 'temp', 'rh', 'wind', 'rain', 'heat_index']
for i, col in enumerate(cols_to_scale):
    df[col] = (df[col] - scaler.mean_[i]) / scaler.scale_[i]

In [90]:
df

,pm25,pm10,o3,co,no2,so2,temp,rh,rain,wind,hour_sin,hour_cos,month_sin,month_cos,heat_index,festival_flag,rush_hour_flag,season_flag,crop_burning_flag
0,-0.066974,-0.630524,2.691880,-0.535910,-0.514159,1.938840,0.374039,0.904355,-0.289896,0.906413,-7.071068e-01,7.071068e-01,1.224647e-16,-1.0,-1.015726,0,0,2,0
1,-0.005459,-0.536116,2.578795,-0.567901,-0.562090,1.518763,0.341279,1.073705,-0.289896,0.857449,-5.000000e-01,8.660254e-01,1.224647e-16,-1.0,-1.031127,0,0,2,0
2,0.022860,-0.483800,2.578795,-0.598010,-0.618009,1.203705,0.395878,1.116043,-0.289896,0.655474,-2.588190e-01,9.659258e-01,1.224647e-16,-1.0,-1.114492,0,0,2,0
3,0.025659,-0.461953,2.635338,-0.627179,-0.705882,0.993667,0.363119,1.116043,-0.289896,0.606510,0.000000e+00,1.000000e+00,1.224647e-16,-1.0,-1.072357,0,0,2,0
4,0.003100,-0.474049,2.748423,-0.653524,-0.801744,0.818634,0.352199,1.073705,-0.289896,0.790124,2.588190e-01,9.659258e-01,1.224647e-16,-1.0,-1.045142,0,0,2,0
5,-0.008324,-0.486248,2.861509,-0.663875,-0.865651,0.748622,0.330359,1.158380,-0.289896,0.312727,5.000000e-01,8.660254e-01,1.224647e-16,-1.0,-1.042962,0,0,2,0
6,-0.008324,-0.483800,2.804966,-0.646938,-0.833698,0.783628,0.308519,1.243055,-0.289896,0.545305,7.071068e-01,7.071068e-01,1.224647e-16,-1.0,-1.038935,0,0,2,0
7,0.017244,-0.457143,2.691880,-0.614006,-0.753813,0.888647,0.286679,1.285393,-0.289896,0.398414,8.660254e-01,5.000000e-01,1.224647e-16,-1.0,-1.022223,0,0,2,0
8,0.072357,-0.384595,3.370395,-0.578251,-0.889617,1.133692,0.341279,1.285393,-0.289896,0.355570,9.659258e-01,2.588190e-01,1.224647e-16,-1.0,-1.092457,0,0,2,0
9,0.080430,-0.359714,4.275081,-0.535910,-1.017432,1.553769,0.439558,1.158380,-0.289896,0.331089,1.000000e+00,6.123234e-17,1.224647e-16,-1.0,-1.183569,0,0,2,0


In [91]:
df.shape

(23, 19)

In [93]:
X_live = df[FEATURE_COLS].values          
X_live = X_live.reshape(1, 23, 19)               

y_pred_scaled = model.predict(X_live).flatten()[0]

aqi_mean  = scaler.mean_[11]
aqi_scale = scaler.scale_[11]
aqi_pred  = y_pred_scaled * aqi_scale + aqi_mean

print(f"Predicted AQI: {aqi_pred:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Predicted AQI: 82.19
